In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import time
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import pickle as pkl
import networkx as nx
import matplotlib.pyplot as plt

import good
from good.reload import deep_reload

In [ ]:
'''
Loading WECC Graph
'''
deep_reload(good)

wecc = good.graph.graph_from_json('Examples/WECC.json')

for source, node in wecc._node.items():
    for asset in node.get('assets', []):
        if asset['type'] == 'load':

            asset['shift_portion'] = .1

In [ ]:
'''
Building from graph
'''
deep_reload(good)

kw = {
    'verbose': True,
    'steps': 25,
    'shortfall_capacity': np.inf,
    'shortfall_cost': 1e3,
    'wastage_capacity': np.inf,
    'wastage_cost': 1e3,
}

network = good.optimization.network.Network(**kw).from_graph(wecc)

'''
Building the model
'''

network.build()

In [ ]:
'''
Solving the model
'''
deep_reload(good)

kw = {
    'solver': {
        '_name': 'appsi_highs',
    },
}

network.solve(**kw)

In [ ]:
{k: v[0] / 1e6 for k, v in network.results.items() if 'capex' in k and v[0] > 0}

In [ ]:
{k: np.array(v) for k, v in network.results.items() if 'shift' in k and 'base' in k}

In [ ]:
{k: np.array(v) for k, v in network.results.items() if 'profile' in k and 'base' in k}

In [ ]:
{k: np.array(v) / 1e6 for k, v in network.results.items() if 'profile' in k}

In [ ]:
{k: np.array(v) / 1e6 for k, v in network.results.items() if 'production' in k and np.array(v).sum() > 0}

In [ ]:
{k: np.array(v) / 1e6 for k, v in network.results.items() if 'consumption' in k and np.array(v).sum() > 0}

In [ ]:
{k: np.array(v) / 1e6 for k, v in network.results.items() if 'level' in k and np.array(v).sum() > 0}

In [ ]:
{k: np.array(v) / 1e6 for k, v in network.results.items() if 'transmission' in k and np.array(v).sum() > 0}

In [ ]:
{k: np.array(v) / 3.6e9 for k, v in network.results.items() if 'shortfall' in k and np.array(v).sum() > 0}

In [ ]:
{k: np.array(v) / 3.6e9 for k, v in network.results.items() if 'wastage' in k and np.array(v).sum() > 0}